# 03 · Evaluating the Agent (four dimensions)

The bootcamp's focus. We score the agent on four dimensions, using the cheapest reliable
tool for each:

| Dimension | Type | What it checks |
|---|---|---|
| Classification accuracy | **rules** | predicted category vs the dataset label |
| Retrieval quality | **rules** | did it cite the correct policy? (match / precision / coverage) |
| Groundedness | **LLM judge** (trace) | does the resolution stick to the retrieved policy? |
| Response quality | **LLM judge** (item) | tone, completeness, actionability (rubric) |
| Efficiency | **trace** | tool calls, latency, cost |

> Rules where there is a ground-truth answer; an LLM judge only where quality is a judgment
> call. The deterministic graders also act as an anchor to sanity-check the judge.

## 1. Rules-based graders (no model needed)\n\nThese are pure functions — instant and reproducible.

In [ ]:
from aieng.agent_evals.complaint_resolution.graders import item_level_deterministic_grader


output = {"predicted_category": "credit_card", "cited_policy_ids": "POL-CREDIT-CARD"}
expected = {"category": "credit_card", "gold_policy_id": "POL-CREDIT-CARD"}

evals = item_level_deterministic_grader(input="", output=output, expected_output=expected)
{e.name: e.value for e in evals}

## 2. Upload a balanced evaluation set to Langfuse

`run_experiment` iterates a Langfuse **dataset**: structured items (input + expected output),
not the raw CSV. Upload a balanced sample first.

In [ ]:
from implementations.complaint_resolution.data.langfuse_upload import upload_complaints_to_langfuse


await upload_complaints_to_langfuse(dataset_name="Complaint-Resolution-Demo", per_category=4)

## 3. Run the four-dimension experiment

This runs the agent over the dataset and applies all graders. You can also run it from the
CLI:

```bash
uv run --env-file .env python implementations/complaint_resolution/evaluate.py \
    --dataset-name Complaint-Resolution-Demo
```

In [ ]:
from aieng.agent_evals.complaint_resolution.agent import create_complaint_resolution_agent
from aieng.agent_evals.complaint_resolution.graders import (
    item_level_deterministic_grader,
    run_level_grader,
)
from aieng.agent_evals.complaint_resolution.retrieval_tool import RETRIEVE_POLICY_TOOL_NAME
from aieng.agent_evals.complaint_resolution.task import ComplaintResolutionTask
from aieng.agent_evals.evaluation import TraceWaitConfig
from aieng.agent_evals.evaluation.experiment import run_experiment_with_trace_evals
from aieng.agent_evals.evaluation.graders import (
    create_llm_as_judge_evaluator,
    create_trace_groundedness_evaluator,
)
from aieng.agent_evals.misalignment_qa.evaluation.hard_metrics import create_trace_usage_evaluator


RUBRIC = "implementations/complaint_resolution/rubrics/resolution_quality.md"


def is_retrieval_obs(obs):
    return RETRIEVE_POLICY_TOOL_NAME in (getattr(obs, "name", "") or "").lower()


resolution_quality = create_llm_as_judge_evaluator(name="resolution_quality", rubric_markdown=RUBRIC)
groundedness = create_trace_groundedness_evaluator(name="groundedness", tool_observation_predicate=is_retrieval_obs)
usage = create_trace_usage_evaluator(name="trace_usage", metrics={"tool_call_count": True, "latency_sec": True})

results = run_experiment_with_trace_evals(
    dataset_name="Complaint-Resolution-Demo",
    name="Complaint Resolution Evaluation (notebook)",
    task=ComplaintResolutionTask(agent=create_complaint_resolution_agent()),
    evaluators=[item_level_deterministic_grader, resolution_quality],
    trace_evaluators=[groundedness, usage],
    run_evaluators=[run_level_grader],
    max_concurrency=2,
    trace_wait=TraceWaitConfig(max_wait_sec=300),
)

## 4. Read the results

In [ ]:
# Run-level classification metrics
for e in results.experiment.run_evaluations:
    if e.name != "category_confusion_matrix":
        print(f"{e.name:28s} {e.value:.3f}")

## Takeaways

- **Classification & retrieval** are graded by rules → trustworthy, reproducible numbers.
- **Groundedness & response quality** need an LLM judge → they catch the things rules can't.
- The retrieval rule ("did it fetch the right policy?") pairs with the groundedness judge
  ("did it stick to that policy?") to catch agents that retrieve correctly but then hallucinate.

All scores are uploaded to Langfuse for comparison across runs.